# Model Training & Export Pipeline

## Overview
This notebook covers the machine learning phase of the project.

### Pipeline Steps:
1. **Load Processed Data**: Import the feature-rich dataset created in the previous ETL step.
2. **Train-Test Split**: Prepare data for validation.
3. **Model Training**: Train an XGBoost Regressor to predict target returns.
4. **Evaluation**: Assess performance using RMSE and MAE.
5. **Model Serialization**: Save the trained model as a `.pkl` file for inference in the Web App.

---

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

print("Libraries loaded.")

## 1. Load Data
We load the `training_data.csv` generated by our ETL pipeline.

In [ ]:
DATA_PATH = "data/processed/training_data.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Please run the ETL notebook first to generate training data.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")

## 2. Prepare Training Set
We select our features (X) and target (y). Note that we exclude non-predictive columns like Date and the Target itself from X.

In [ ]:
# Define features to use
FEATURE_COLS = [
    'SMA_20', 'SMA_50', 'EMA_12', 
    'RSI', 'MACD', 'MACD_signal', 
    'BB_upper', 'BB_lower', 'ATR', 
    'Volume_Change', 'Prev_Close', 'Sentiment_Score'
]
TARGET_COL = 'Target_Return'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

# Split: 80% Train, 20% Evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training Set: {X_train.shape}")
print(f"Test Set: {X_test.shape}")

## 3. Train Model (XGBoost)
We use XGBoost for its efficiency and performance on structured data. We configure it for regression (predicting continuous return).

In [ ]:
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

print("Training model...")
model.fit(X_train, y_train)
print("Training complete.")

## 4. Evaluation
We evaluate the model using RMSE (Root Mean Squared Error) on the unseen test set.

In [ ]:
predictions = model.predict(X_test)

mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)

print(f"Model Performance:")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")

# Plot Actual vs Predicted
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:100], label='Actual', alpha=0.7)
plt.plot(predictions[:100], label='Predicted', alpha=0.7)
plt.title("Actual vs Predicted Returns (First 100 Test Samples)")
plt.legend()
plt.show()

## 5. Model Serialization (Data Export)
Finally, we save the trained model to a `.pkl` file. This file will be loaded by the FastAPI backend to serve real-time predictions.

In [ ]:
MODEL_DIR = "models"
MODEL_FILENAME = "xgb_model_global.pkl"
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_FILENAME)

os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, MODEL_PATH)

print(f"Model saved successfully to: {MODEL_PATH}")